Questo notebookl andalizza il dataset costruito nel Notebook01 per capire se è adatto alla costruzione di un sistema di raccomandazione.
In particolare analizzare la sparsità dei dati.

In [3]:
#Se mancano pacchetti, esegui questa cella una sola volta.
%pip install -q "datasets==3.6.0" pandas pyarrow requests tqdm
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 3.6 MB/s eta 0:00:03
   ------ --------------------------------- 1.3/8.2 MB 4.0 MB/s eta 0:00:02
   ------------ --------------------------- 2.6/8.2 MB 4.7 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.2 MB 5.6 MB/s eta 0:00:01
   ----------------------------- ---------- 6.0/8.2 MB 6.5 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 7.1 MB/s  0:00:01
Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   --------


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from pathlib import Path
import json
import math
import warnings #gestire warnings
import numpy as np 
import pandas as pd
import pyarrow.parquet as pq 
import matplotlib.pyplot as plt 

warnings.filterwarnings("ignore", category=FutureWarning) #nasconde avvisi per output meno rumoroso
#impostazioni di pandas
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")  

In [ ]:
#percorsi
CATEGORY="Electronics"

PROJECT_ROOT = Path("..").resolve() if (Path("..") / "data").exists() else Path(".").resolve()  #determina automaticamente la cartella principale del progetto
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "electronics_intersection" #percorso dati notebook01
ANALYSIS_DIR = PROCESSED_DIR / "analysis" #dove salvare risultati notebook02
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True) #la crea se non esiste già

#percorsi dei file prodotti nel notebook01
ITEMS_PATH = PROCESSED_DIR / "items.parquet"
QA_PATH = PROCESSED_DIR / "qa.parquet"
RATINGS_PATH = PROCESSED_DIR / "ratings.parquet"
SUMMARY_PATH = PROCESSED_DIR / "summary.json"

#dizionario per associare nome leggibile ad ogni percorso
paths={
    "items": ITEMS_PATH,
    "qa": QA_PATH,
    "ratings":RATINGS_PATH,
    "summary": SUMMARY_PATH,
}
#controllo dei percorsi
print("Project root:", PROJECT_ROOT)
print("Processed dir:", PROCESSED_DIR)
print("Analysis dir:", ANALYSIS_DIR)
for name, path in paths.items():
    print(f"{name:8s}->{path} | exists={path.exists()}")

Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Processed dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection
Analysis dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection\analysis
items   ->C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection\items.parquet | exists=True
qa      ->C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection\qa.parquet | exists=True
ratings ->C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection\ratings.parquet | exists=True
summary ->C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection\summary.json | exists=True


In [ ]:
#dati attesi mancanti, in caso avvisa di eseguire prima notebook01
missing =[str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "mancano cerci file del notebook01: \n"
        + "\n".join(missing)+ "\n\nEsegui prima il notebook: 01_prepare_electronics_intersection.ipynb"
    )

In [7]:
#lettura dei metadata dei parquet
def parquet_info(path: Path)->dict:
    pf=pq.ParquetFile(path)
    return{
        "file":path.name,
        "rows":pf.metadata.num_rows,
        "row_groups":pf.metadata.num_row_groups,
        "columns": len(pf.schema_arrow.names),
        "column_names": ", ".join(pf.schema_arrow.names),
    }

parquet_infos=pd.DataFrame([
    parquet_info(ITEMS_PATH),
    parquet_info(QA_PATH),
    parquet_info(RATINGS_PATH),
])

parquet_infos

,file,rows,row_groups,columns,column_names
0,items.parquet,22974,1,12,"parent_asin, title, category, avg_rating, num_ratings, price, store, features_json, description_json, categories_jso..."
1,qa.parquet,180976,1,8,"qa_id, parent_asin, question, answer, questionType, answerType, answerTime, unixTime"
2,ratings.parquet,3484401,18,6,"user_id, parent_asin, asin, rating, timestamp, verified_purchase"


In [8]:
#controlliamo summary.json
if SUMMARY_PATH.exists():
    summary=json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    display(pd.DataFrame([summary]).T.rename(columns={0: "rows_from_summary_json"}))
else:
    summary={}
    print("summary.json non trovato. Ricarica notebook 02")

,rows_from_summary_json
items,22974
qa,180976
ratings,3484401
